In [3]:
# ============================================================
# FINAL MODEL TRAINING
# Random Forest Classifier + Random Forest Regressor
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

FILE_PATH = "../data/raw/CPRI_Hackathon_Screening_Dataset_PARTICIPANT.xlsx"

train = pd.read_excel(
    FILE_PATH,
    sheet_name="Training_Data"
)

test = pd.read_excel(
    FILE_PATH,
    sheet_name="Test_Data"
)

print("Training shape:", train.shape)
print("Test shape    :", test.shape)


# ============================================================
# 2. Feature Engineering
# ============================================================

def create_features(df):

    df = df.copy()

    # --------------------------------------------------------
    # Sensor relationship features
    # --------------------------------------------------------

    df["S2_S1_residual"] = (
        df["Sensor_S2"] - df["Sensor_S1"]
    )

    df["S3_S1_residual"] = (
        df["Sensor_S3"] - df["Sensor_S1"]
    )

    df["S3_S2_residual"] = (
        df["Sensor_S3"] - df["Sensor_S2"]
    )

    # Absolute sensor inconsistencies
    df["S2_S1_residual_abs"] = (
        df["S2_S1_residual"].abs()
    )

    df["S3_S1_residual_abs"] = (
        df["S3_S1_residual"].abs()
    )

    df["S3_S2_residual_abs"] = (
        df["S3_S2_residual"].abs()
    )

    # Sensor differences
    df["S2_minus_S1"] = (
        df["Sensor_S2"] - df["Sensor_S1"]
    )

    df["S3_minus_S1"] = (
        df["Sensor_S3"] - df["Sensor_S1"]
    )

    df["S3_minus_S2"] = (
        df["Sensor_S3"] - df["Sensor_S2"]
    )

    # --------------------------------------------------------
    # Operating-condition interaction
    # --------------------------------------------------------

    df["Voltage_Current"] = (
        df["Applied_Voltage_kV"] *
        df["Load_Current_A"]
    )

    # --------------------------------------------------------
    # Missingness indicators
    # --------------------------------------------------------

    df["Sensor_S1_missing"] = (
        df["Sensor_S1"].isna().astype(int)
    )

    df["Sensor_S2_missing"] = (
        df["Sensor_S2"].isna().astype(int)
    )

    df["Sensor_S3_missing"] = (
        df["Sensor_S3"].isna().astype(int)
    )

    df["Sensor_S4_missing"] = (
        df["Sensor_S4"].isna().astype(int)
    )

    return df


train_features = create_features(train)
test_features = create_features(test)


# ============================================================
# 3. Define FINAL feature set
# ============================================================

FEATURES = [
    "Applied_Voltage_kV",
    "Load_Current_A",
    "Ambient_Temperature_C",
    "Test_Duration_min",
    "Sensor_S1",
    "Sensor_S2",
    "Sensor_S3",
    "Sensor_S4",

    # Sensor consistency
    "S2_S1_residual",
    "S3_S1_residual",
    "S3_S2_residual",

    "S2_S1_residual_abs",
    "S3_S1_residual_abs",
    "S3_S2_residual_abs",

    "S2_minus_S1",
    "S3_minus_S1",
    "S3_minus_S2",

    # Operating condition
    "Voltage_Current",

    # Missingness
    "Sensor_S1_missing",
    "Sensor_S2_missing",
    "Sensor_S3_missing",
    "Sensor_S4_missing"
]


X_train_final = train_features[FEATURES].copy()
X_test_final = test_features[FEATURES].copy()


print("\nFinal feature count:", len(FEATURES))
print("Training matrix   :", X_train_final.shape)
print("Test matrix       :", X_test_final.shape)


# ============================================================
# 4. Imputation
# ============================================================
# Fit ONLY on the complete training dataset.
# Then apply the same transformation to Test_Data.

imputer = SimpleImputer(strategy="median")

X_train_final = imputer.fit_transform(
    X_train_final
)

X_test_final = imputer.transform(
    X_test_final
)


print("\nMissing values after imputation:")
print(
    "Training:",
    np.isnan(X_train_final).sum()
)

print(
    "Test    :",
    np.isnan(X_test_final).sum()
)


# ============================================================
# 5. Prepare targets
# ============================================================

# Classification
# Valid   = 0
# Invalid = 1

y_class_final = (
    train["Validity_Label"] == "Invalid"
).astype(int)


# Regression
y_reg_final = train["Reference_Parameter"]


print("\nTarget distributions:")
print(
    pd.Series(y_class_final)
    .map({
        0: "Valid",
        1: "Invalid"
    })
    .value_counts()
)


# ============================================================
# 6. FINAL RANDOM FOREST CLASSIFIER
# ============================================================

final_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    max_features=0.5,
    min_samples_leaf=1,
    min_samples_split=2,

    class_weight="balanced",

    random_state=42,
    n_jobs=-1
)


print("\nTraining final RF classifier...")

final_classifier.fit(
    X_train_final,
    y_class_final
)


print("Final classifier trained.")
print("Classes:", final_classifier.classes_)


# ============================================================
# 7. FINAL RANDOM FOREST REGRESSOR
# ============================================================

final_regressor = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,

    random_state=42,
    n_jobs=-1
)


print("\nTraining final RF regressor...")

final_regressor.fit(
    X_train_final,
    y_reg_final
)


print("Final regressor trained.")


# ============================================================
# 8. Training sanity check
# ============================================================

train_reg_pred = final_regressor.predict(
    X_train_final
)

train_class_prob = final_classifier.predict_proba(
    X_train_final
)[:, 1]

train_class_pred = (
    train_class_prob >= 0.415
).astype(int)


print("\n================================================")
print("FINAL MODEL TRAINING COMPLETE")
print("================================================")

print("\nRegression — Training")
print("----------------------")

print(
    f"MAE  : "
    f"{mean_absolute_error(y_reg_final, train_reg_pred):.4f}"
)

print(
    f"RMSE : "
    f"{np.sqrt(mean_squared_error(y_reg_final, train_reg_pred)):.4f}"
)

print(
    f"R²   : "
    f"{r2_score(y_reg_final, train_reg_pred):.4f}"
)


print("\nClassification — Training")
print("--------------------------")

print(
    "Threshold:",
    0.415
)

print(
    pd.Series(train_class_pred)
    .map({
        0: "Valid",
        1: "Invalid"
    })
    .value_counts()
)


# ============================================================
# 9. TEST PREDICTIONS
# ============================================================

# Reference Parameter prediction
test_reference_prediction = final_regressor.predict(
    X_test_final
)


# Invalid probability
test_invalid_probability = final_classifier.predict_proba(
    X_test_final
)[:, 1]


# Apply optimized threshold
test_class_prediction = (
    test_invalid_probability >= 0.415
).astype(int)


# Convert back to required labels
test_validity_prediction = np.where(
    test_class_prediction == 1,
    "Invalid",
    "Valid"
)


# ============================================================
# 10. Create submission dataframe
# ============================================================

final_submission = pd.DataFrame({

    "Test_ID": test["Test_ID"],

    "Predicted_Reference_Parameter":
        test_reference_prediction,

    "Validity_Label":
        test_validity_prediction
})


print("\nFINAL TEST PREDICTIONS")
print("======================")

print("Number of predictions:", len(final_submission))

print("\nValidity distribution:")

print(
    final_submission["Validity_Label"]
    .value_counts()
)

print("\nReference Parameter statistics:")

print(
    final_submission[
        "Predicted_Reference_Parameter"
    ].describe()
)


# ============================================================
# 11. Preview
# ============================================================

display(
    final_submission.head(10)
)

Training shape: (1000, 11)
Test shape    : (350, 9)

Final feature count: 22
Training matrix   : (1000, 22)
Test matrix       : (350, 22)

Missing values after imputation:
Training: 0
Test    : 0

Target distributions:
Validity_Label
Valid      866
Invalid    134
Name: count, dtype: int64

Training final RF classifier...
Final classifier trained.
Classes: [0 1]

Training final RF regressor...
Final regressor trained.

FINAL MODEL TRAINING COMPLETE

Regression — Training
----------------------
MAE  : 0.4133
RMSE : 1.3148
R²   : 0.9849

Classification — Training
--------------------------
Threshold: 0.415
Valid      864
Invalid    136
Name: count, dtype: int64

FINAL TEST PREDICTIONS
Number of predictions: 350

Validity distribution:
Validity_Label
Valid      310
Invalid     40
Name: count, dtype: int64

Reference Parameter statistics:
count    350.000000
mean      26.215776
std        9.996892
min       13.515772
25%       18.565490
50%       22.861776
75%       32.114358
max       55.4

,Test_ID,Predicted_Reference_Parameter,Validity_Label
0,TST-0278,32.540006,Invalid
1,TST-0006,19.109109,Valid
2,TST-0047,52.900732,Valid
3,TST-0311,25.466407,Valid
4,TST-0264,18.582008,Invalid
5,TST-0169,19.134810,Valid
6,TST-0269,18.371422,Valid
7,TST-0178,16.475410,Invalid
8,TST-0206,31.636522,Valid
9,TST-0038,17.953792,Valid


In [4]:
# ============================================================
# FINAL SUBMISSION GENERATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TEAM_NAME = "0xPOWER"  

SUBMISSION_DIR = "../submissions"

os.makedirs(SUBMISSION_DIR, exist_ok=True)


# ------------------------------------------------------------
# 2. Build final submission dataframe
# ------------------------------------------------------------

final_submission = pd.DataFrame({
    "Test_ID": test["Test_ID"].values,

    "Predicted_Reference_Parameter":
        test_reference_prediction,

    "Validity_Label":
        test_validity_prediction
})


# ------------------------------------------------------------
# 3. Round predicted Reference Parameter
# ------------------------------------------------------------

final_submission[
    "Predicted_Reference_Parameter"
] = final_submission[
    "Predicted_Reference_Parameter"
].round(4)


# ------------------------------------------------------------
# 4. Save CSV
# ------------------------------------------------------------

csv_path = os.path.join(
    SUBMISSION_DIR,
    f"{TEAM_NAME}.csv"
)

final_submission.to_csv(
    csv_path,
    index=False
)


# ------------------------------------------------------------
# 5. Calculate summary statistics
# ------------------------------------------------------------

total_records = len(final_submission)

invalid_count = (
    final_submission["Validity_Label"]
    .eq("Invalid")
    .sum()
)

valid_count = (
    final_submission["Validity_Label"]
    .eq("Valid")
    .sum()
)

predicted_reference = (
    final_submission[
        "Predicted_Reference_Parameter"
    ]
)


# ------------------------------------------------------------
# 6. Identify highest-attention records
# ------------------------------------------------------------
#
# Attention is based on the model's probability that a record
# is Invalid.
#
# Higher Invalid probability = higher attention.
#
# This is more defensible than simply selecting records with
# the highest predicted temperature.
# ------------------------------------------------------------

attention_df = pd.DataFrame({
    "Test_ID": test["Test_ID"].values,
    "Invalid_Probability": test_invalid_probability,
    "Validity_Label": test_validity_prediction
})


top_attention = (
    attention_df
    .sort_values(
        "Invalid_Probability",
        ascending=False
    )
    .head(3)
)


top_attention_ids = (
    top_attention["Test_ID"]
    .tolist()
)


# ------------------------------------------------------------
# 7. Create explanation
# ------------------------------------------------------------

explanation = (
    f"The model predicts {invalid_count} of {total_records} "
    f"test records as Invalid. Classification is based on "
    f"sensor relationships, operating conditions, and missingness "
    f"patterns identified during training. Reference_Parameter "
    f"is predicted using a Random Forest regression model. "
    f"The three highest-attention records are those with the "
    f"highest predicted probability of being Invalid."
)


# ------------------------------------------------------------
# 8. Create summary JSON
# ------------------------------------------------------------

summary = {
    "record_count": int(total_records),

    "abnormal_invalid_count": int(invalid_count),

    "valid_count": int(valid_count),

    "predicted_reference_parameter": {
        "minimum": float(predicted_reference.min()),
        "maximum": float(predicted_reference.max()),
        "average": float(predicted_reference.mean())
    },

    "top_3_test_ids_requiring_highest_attention":
        top_attention_ids,

    "attention_method":
        "Ranked by predicted probability of Invalid classification.",

    "explanation": explanation
}


# ------------------------------------------------------------
# 9. Save summary.json
# ------------------------------------------------------------

json_path = os.path.join(
    SUBMISSION_DIR,
    "summary.json"
)


with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ============================================================
# 10. Display final results
# ============================================================

print("================================================")
print("SUBMISSION FILES GENERATED")
print("================================================")

print(f"\nCSV     : {csv_path}")
print(f"Summary : {json_path}")

print("\nSubmission shape:")
print(final_submission.shape)

print("\nValidity distribution:")
print(
    final_submission["Validity_Label"]
    .value_counts()
)

print("\nReference Parameter statistics:")
print(
    predicted_reference.describe()
)

print("\nTOP 3 HIGH-ATTENTION RECORDS")
print("============================")

display(top_attention)

print("\nSummary JSON:")
print(
    json.dumps(
        summary,
        indent=4
    )
)


# ------------------------------------------------------------
# 11. Final sanity checks
# ------------------------------------------------------------

assert len(final_submission) == len(test)

assert len(final_submission) == 350

assert final_submission["Test_ID"].isna().sum() == 0

assert (
    final_submission[
        "Predicted_Reference_Parameter"
    ].isna().sum()
    == 0
)

assert (
    final_submission["Validity_Label"]
    .isin(["Valid", "Invalid"])
    .all()
)

assert len(top_attention_ids) == 3

print("\nALL SANITY CHECKS PASSED.")

SUBMISSION FILES GENERATED

CSV     : ../submissions\0xPOWER.csv
Summary : ../submissions\summary.json

Submission shape:
(350, 3)

Validity distribution:
Validity_Label
Valid      310
Invalid     40
Name: count, dtype: int64

Reference Parameter statistics:
count    350.000000
mean      26.215776
std        9.996889
min       13.515800
25%       18.565500
50%       22.861750
75%       32.114350
max       55.466000
Name: Predicted_Reference_Parameter, dtype: float64

TOP 3 HIGH-ATTENTION RECORDS


,Test_ID,Invalid_Probability,Validity_Label
7,TST-0178,1.0,Invalid
4,TST-0264,1.0,Invalid
29,TST-0222,1.0,Invalid



Summary JSON:
{
    "record_count": 350,
    "abnormal_invalid_count": 40,
    "valid_count": 310,
    "predicted_reference_parameter": {
        "minimum": 13.5158,
        "maximum": 55.466,
        "average": 26.215775714285712
    },
    "top_3_test_ids_requiring_highest_attention": [
        "TST-0178",
        "TST-0264",
        "TST-0222"
    ],
    "attention_method": "Ranked by predicted probability of Invalid classification.",
    "explanation": "The model predicts 40 of 350 test records as Invalid. Classification is based on sensor relationships, operating conditions, and missingness patterns identified during training. Reference_Parameter is predicted using a Random Forest regression model. The three highest-attention records are those with the highest predicted probability of being Invalid."
}

ALL SANITY CHECKS PASSED.


In [5]:
# ============================================================
# FINAL SUBMISSION QA
# ============================================================

import os
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TEAM_NAME = "0xPOWER"

SUBMISSION_DIR = "../submissions"

CSV_PATH = os.path.join(
    SUBMISSION_DIR,
    f"{TEAM_NAME}.csv"
)

JSON_PATH = os.path.join(
    SUBMISSION_DIR,
    "summary.json"
)


# ============================================================
# 2. Check files exist
# ============================================================

print("=" * 60)
print("1. FILE EXISTENCE")
print("=" * 60)

assert os.path.exists(CSV_PATH), (
    f"Submission CSV not found: {CSV_PATH}"
)

assert os.path.exists(JSON_PATH), (
    f"Summary JSON not found: {JSON_PATH}"
)

print("✓ Submission CSV exists")
print("✓ summary.json exists")


# ============================================================
# 3. Load submission files
# ============================================================

submission_qa = pd.read_csv(CSV_PATH)

with open(
    JSON_PATH,
    "r",
    encoding="utf-8"
) as f:
    summary_qa = json.load(f)

print("\nFiles loaded successfully.")


# ============================================================
# 4. Check CSV shape
# ============================================================

print("\n" + "=" * 60)
print("2. CSV SHAPE")
print("=" * 60)

print("Shape:", submission_qa.shape)

assert len(submission_qa) == 350, (
    f"Expected 350 rows, found {len(submission_qa)}"
)

assert submission_qa.shape[1] == 3, (
    f"Expected 3 columns, found {submission_qa.shape[1]}"
)

print("✓ Exactly 350 prediction rows")
print("✓ Exactly 3 columns")


# ============================================================
# 5. Check exact column names
# ============================================================

print("\n" + "=" * 60)
print("3. COLUMN FORMAT")
print("=" * 60)

expected_columns = [
    "Test_ID",
    "Predicted_Reference_Parameter",
    "Validity_Label"
]

print("Expected:")
print(expected_columns)

print("\nActual:")
print(submission_qa.columns.tolist())

assert submission_qa.columns.tolist() == expected_columns, (
    "Column names/order do not match expected format."
)

print("\n✓ Column names and order are correct")


# ============================================================
# 6. Check Test IDs
# ============================================================

print("\n" + "=" * 60)
print("4. TEST ID VALIDATION")
print("=" * 60)

test_ids = test["Test_ID"].astype(str)
submission_ids = submission_qa["Test_ID"].astype(str)

# No missing IDs
assert submission_ids.isna().sum() == 0

# No duplicate IDs
assert submission_ids.duplicated().sum() == 0, (
    "Duplicate Test_IDs found in submission."
)

# Same IDs as Test_Data
assert set(submission_ids) == set(test_ids), (
    "Submission Test_IDs do not exactly match Test_Data."
)

# Same order
assert submission_ids.tolist() == test_ids.tolist(), (
    "Submission Test_ID order does not match Test_Data."
)

print("✓ No missing Test_IDs")
print("✓ No duplicate Test_IDs")
print("✓ All 350 Test_IDs match Test_Data")
print("✓ Test_ID order is correct")


# ============================================================
# 7. Check Reference Parameter predictions
# ============================================================

print("\n" + "=" * 60)
print("5. REGRESSION PREDICTION QA")
print("=" * 60)

reference_pred = submission_qa[
    "Predicted_Reference_Parameter"
]

# No missing values
assert reference_pred.isna().sum() == 0, (
    "Missing Reference_Parameter predictions found."
)

# Must be numeric
assert pd.api.types.is_numeric_dtype(
    reference_pred
), "Reference predictions are not numeric."

# Must be finite
assert np.isfinite(
    reference_pred
).all(), "Non-finite Reference predictions found."

# No negative values
assert (
    reference_pred >= 0
).all(), "Negative Reference predictions found."

print("✓ No missing predictions")
print("✓ Predictions are numeric")
print("✓ All predictions are finite")
print("✓ No negative predictions")

print("\nPrediction statistics:")
print(
    reference_pred.describe()
)


# ============================================================
# 8. Check classification labels
# ============================================================

print("\n" + "=" * 60)
print("6. CLASSIFICATION QA")
print("=" * 60)

allowed_labels = {
    "Valid",
    "Invalid"
}

actual_labels = set(
    submission_qa["Validity_Label"].unique()
)

print("Labels found:", actual_labels)

assert actual_labels.issubset(
    allowed_labels
), (
    f"Unexpected labels found: "
    f"{actual_labels - allowed_labels}"
)

assert submission_qa[
    "Validity_Label"
].isna().sum() == 0, (
    "Missing validity labels found."
)

print("✓ Only Valid/Invalid labels present")
print("✓ No missing validity labels")

print("\nValidity distribution:")
print(
    submission_qa[
        "Validity_Label"
    ].value_counts()
)


# ============================================================
# 9. Check summary JSON
# ============================================================

print("\n" + "=" * 60)
print("7. SUMMARY JSON QA")
print("=" * 60)

# Required keys
required_summary_keys = [
    "record_count",
    "abnormal_invalid_count",
    "valid_count",
    "predicted_reference_parameter",
    "top_3_test_ids_requiring_highest_attention",
    "attention_method",
    "explanation"
]

for key in required_summary_keys:

    assert key in summary_qa, (
        f"Missing summary key: {key}"
    )

print("✓ All required summary fields exist")


# Record count
assert summary_qa[
    "record_count"
] == 350

print("✓ Record count = 350")


# Invalid count
actual_invalid_count = int(
    (
        submission_qa[
            "Validity_Label"
        ] == "Invalid"
    ).sum()
)

assert summary_qa[
    "abnormal_invalid_count"
] == actual_invalid_count

print(
    f"✓ Invalid count = {actual_invalid_count}"
)


# Valid count
actual_valid_count = int(
    (
        submission_qa[
            "Validity_Label"
        ] == "Valid"
    ).sum()
)

assert summary_qa[
    "valid_count"
] == actual_valid_count

print(
    f"✓ Valid count = {actual_valid_count}"
)


# ============================================================
# 10. Validate summary statistics
# ============================================================

print("\n" + "=" * 60)
print("8. SUMMARY STATISTICS QA")
print("=" * 60)

summary_stats = summary_qa[
    "predicted_reference_parameter"
]

actual_min = float(reference_pred.min())
actual_max = float(reference_pred.max())
actual_mean = float(reference_pred.mean())

assert np.isclose(
    summary_stats["minimum"],
    actual_min,
    atol=1e-4
)

assert np.isclose(
    summary_stats["maximum"],
    actual_max,
    atol=1e-4
)

assert np.isclose(
    summary_stats["average"],
    actual_mean,
    atol=1e-4
)

print("✓ Minimum matches CSV")
print("✓ Maximum matches CSV")
print("✓ Average matches CSV")


# ============================================================
# 11. Validate top-3 attention IDs
# ============================================================

print("\n" + "=" * 60)
print("9. ATTENTION RECORD QA")
print("=" * 60)

top3 = summary_qa[
    "top_3_test_ids_requiring_highest_attention"
]

assert len(top3) == 3

assert len(set(top3)) == 3

assert all(
    test_id in set(test_ids)
    for test_id in top3
)

print("✓ Exactly 3 attention records")
print("✓ All 3 IDs exist in Test_Data")

print("\nTop 3:")
for i, test_id in enumerate(top3, 1):
    print(f"{i}. {test_id}")


# ============================================================
# 12. Explanation word count
# ============================================================

print("\n" + "=" * 60)
print("10. EXPLANATION QA")
print("=" * 60)

explanation = summary_qa["explanation"]

word_count = len(
    explanation.split()
)

print("Explanation word count:", word_count)

assert word_count <= 100, (
    f"Explanation exceeds 100 words: {word_count}"
)

print("✓ Explanation is within 100-word limit")


# ============================================================
# 13. Check prediction consistency with notebook
# ============================================================

print("\n" + "=" * 60)
print("11. NOTEBOOK ↔ CSV CONSISTENCY")
print("=" * 60)

# Compare CSV with the predictions currently in memory

assert np.allclose(
    submission_qa[
        "Predicted_Reference_Parameter"
    ].values,
    np.round(
        test_reference_prediction,
        4
    ),
    atol=1e-4
)

assert (
    submission_qa[
        "Validity_Label"
    ].tolist()
    ==
    test_validity_prediction.tolist()
)

print("✓ CSV regression predictions match notebook")
print("✓ CSV classification predictions match notebook")


# ============================================================
# 14. FINAL RESULT
# ============================================================

print("\n")
print("=" * 60)
print("                FINAL QA RESULT")
print("=" * 60)

print("\n✓ FILES EXIST")
print("✓ CSV FORMAT CORRECT")
print("✓ 350 TEST RECORDS PRESENT")
print("✓ TEST IDs VERIFIED")
print("✓ NO DUPLICATE IDs")
print("✓ NO MISSING PREDICTIONS")
print("✓ ALL REGRESSION VALUES FINITE")
print("✓ CLASSIFICATION LABELS VALID")
print("✓ SUMMARY.JSON VERIFIED")
print("✓ SUMMARY STATISTICS VERIFIED")
print("✓ TOP-3 ATTENTION IDS VERIFIED")
print("✓ EXPLANATION ≤ 100 WORDS")
print("✓ CSV ↔ NOTEBOOK PREDICTIONS MATCH")

print("\n" + "=" * 60)
print("          ALL FINAL QA CHECKS PASSED ✓")
print("=" * 60)

1. FILE EXISTENCE
✓ Submission CSV exists
✓ summary.json exists

Files loaded successfully.

2. CSV SHAPE
Shape: (350, 3)
✓ Exactly 350 prediction rows
✓ Exactly 3 columns

3. COLUMN FORMAT
Expected:
['Test_ID', 'Predicted_Reference_Parameter', 'Validity_Label']

Actual:
['Test_ID', 'Predicted_Reference_Parameter', 'Validity_Label']

✓ Column names and order are correct

4. TEST ID VALIDATION
✓ No missing Test_IDs
✓ No duplicate Test_IDs
✓ All 350 Test_IDs match Test_Data
✓ Test_ID order is correct

5. REGRESSION PREDICTION QA
✓ No missing predictions
✓ Predictions are numeric
✓ All predictions are finite
✓ No negative predictions

Prediction statistics:
count    350.000000
mean      26.215776
std        9.996889
min       13.515800
25%       18.565500
50%       22.861750
75%       32.114350
max       55.466000
Name: Predicted_Reference_Parameter, dtype: float64

6. CLASSIFICATION QA
Labels found: {'Valid', 'Invalid'}
✓ Only Valid/Invalid labels present
✓ No missing validity labels

Va